# receipt-gen: Google スプレッドシートから領収書 PDF を作る

スプレッドシートの 1 行を 1 枚の領収書として、PDF を Google Drive に保存します。
どの列に何が入っているかは、Drive に置いた対応関係の JSON（`mapping.json`）で指定します。書き方は `docs/spreadsheet.md` を参照してください。

上から順に実行してください。
1. 設定を書き換えます。
2. Drive をマウントし、receipt-gen をインストールします。
3. Google アカウントでログインします（スプレッドシートの読み取りに使います）。
4. `mapping.json` を準備し、シートの見出しを確認します。
5. 内容をプレビューし、問題がなければ PDF を作ります。

## 1. 設定

In [ ]:
# 読み込むスプレッドシートの URL
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/XXXXXXXX/edit"
# シート（タブ）名。None なら先頭のシート
WORKSHEET = None

# Drive 上の作業フォルダ。mapping.json、印影画像、出力 PDF をここに置きます
DRIVE_DIR = "/content/drive/MyDrive/receipt-gen"
MAPPING_FILE = "mapping.json"

# receipt-gen のコードの取得元
#   "github": GitHub から clone します。非公開リポジトリの場合は、Colab の「シークレット」（左の鍵アイコン）に
#             GITHUB_TOKEN を登録してください
#   "drive" : Drive に置いたリポジトリのフォルダ（CODE_DIR_IN_DRIVE）を使います
CODE_SOURCE = "github"
REPO_URL = "https://github.com/ytuesugi/receipt-gen.git"
REPO_BRANCH = "main"
CODE_DIR_IN_DRIVE = "/content/drive/MyDrive/receipt-gen/code"

## 2. Drive のマウントとインストール

In [ ]:
import os
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

if CODE_SOURCE == "github":
    REPO_DIR = "/content/receipt-gen"
    url = REPO_URL
    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        pass  # 公開リポジトリならトークンは不要
    if os.path.exists(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    else:
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, url, REPO_DIR],
                check=True,
            )
        except subprocess.CalledProcessError:
            # トークンを含む URL を表示しないよう、元の例外は出さない
            raise RuntimeError(
                "git clone に失敗しました。REPO_URL、REPO_BRANCH、GITHUB_TOKEN を確認してください。"
            ) from None
else:
    REPO_DIR = CODE_DIR_IN_DRIVE

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"],
    check=True,
)
sys.path.insert(0, f"{REPO_DIR}/src")
print("receipt-gen:", REPO_DIR)

## 3. Google アカウントでログイン

スプレッドシートを読むために、Colab に Google アカウントへのアクセスを許可します。

In [ ]:
import google.auth
import gspread
from google.colab import auth

auth.authenticate_user()
credentials, _ = google.auth.default()
gc = gspread.authorize(credentials)

## 4. mapping.json の準備

初めて実行したときは、サンプルをもとに `mapping.json` を作業フォルダに作ります。
左のファイル欄で `drive/MyDrive/receipt-gen/mapping.json` をダブルクリックすると、Colab 上で編集できます。
下に表示されるシートの見出しに合わせて、`"column"` の値を書き換えてください。
印影を入れる場合は、画像を同じフォルダに置き、`issuer.stamp_file` にファイル名を書きます。

In [ ]:
import json
from pathlib import Path

drive_dir = Path(DRIVE_DIR)
drive_dir.mkdir(parents=True, exist_ok=True)
mapping_path = drive_dir / MAPPING_FILE

if mapping_path.exists():
    print(f"既存の {mapping_path} を使います。")
else:
    example = json.loads(Path(REPO_DIR, "input/sheet-example.json").read_text(encoding="utf-8"))
    example["source"] = {"type": "google_sheets", "url": SPREADSHEET_URL, "worksheet": WORKSHEET}
    example["issuer"]["stamp_file"] = None
    mapping_path.write_text(json.dumps(example, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"{mapping_path} を作成しました。シートに合わせて編集してから次へ進んでください。")

spreadsheet = gc.open_by_url(SPREADSHEET_URL)
worksheet = spreadsheet.worksheet(WORKSHEET) if WORKSHEET else spreadsheet.sheet1
print(f"シート「{worksheet.title}」の見出し:")
for header in worksheet.row_values(1):
    print(" -", header)

## 5. プレビュー

PDF はまだ作りません。宛名・明細・合計を確認してください。`mapping.json` を直したら、このセルだけ再実行すれば反映されます。

In [ ]:
import pandas as pd

from receipt_gen import ConfigError
from receipt_gen.sheet import build_config_from_mapping, preview_rows

try:
    result = build_config_from_mapping(
        mapping_path,
        source_override=SPREADSHEET_URL,
        worksheet=WORKSHEET,
        gspread_client=gc,
    )
except ConfigError as error:
    raise SystemExit(f"設定またはシートに問題があります: {error}")

for note in result.skipped:
    print("Skipped", note)
preview = pd.DataFrame(preview_rows(result.config))
print(f"{len(preview)} 件、合計 ¥{preview['total'].sum():,}")
preview

## 6. PDF の生成

`DRIVE_DIR/output/日時/` に保存します。

In [ ]:
from receipt_gen import generate_receipts_from_config

output_files = generate_receipts_from_config(
    result.config,
    base_dir=result.base_dir,
    output_root=drive_dir / "output",
    show_progress=True,
)
output_dir = output_files[0].parent
print(f"{len(output_files)} 件を {output_dir} に保存しました。")

## 7. （任意）ZIP でダウンロード

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(f"/content/{output_dir.name}", "zip", output_dir)
files.download(archive)